## Modelagem
### Lógica do Escalonamento Proporcional
Podemos usar o tempo em minutos de cada timeframe como a base do nosso cálculo. A ideia é que o threshold cresça com a raiz quadrada do tempo (já que a volatilidade geralmente escala com raiz de t e o horizonte acompanhe a duração da tendência esperada.

In [0]:
%skip
dbutils.notebook.exit("Detaching")

In [0]:
import pyspark.sql.functions as f
from pyspark.sql.window import Window

df_feat = spark.table('workspace.default.tcripto_features').orderBy("timestamp")
timeframe = '1h'

w = Window.orderBy("timestamp")

In [0]:
df_feat.count()

In [0]:
import math

# Base de tempo
tf_map = {'15m': 15, '1h': 60, '4h': 240}
minutos_base = tf_map.get(timeframe, 60)
peso_agressividade = 0.5

# Barreiras
horizonte = int((360 / minutos_base) * peso_agressividade) # Barreira Vertical (Tempo)
threshold = round(0.001 * math.sqrt(minutos_base) * peso_agressividade, 4) # Barreira Superior (Take Profit)
stop_loss = threshold * 0.5 # Barreira Inferior (Stop Loss) - Ex: Risco/Retorno 2:1

print(f"Horizonte: {horizonte} candles | Take Profit: {threshold*100:.2f}% | Stop Loss: {stop_loss*100:.2f}%")

# Janela olhando para o futuro
w_forward = Window.orderBy("timestamp").rowsBetween(1, horizonte)

# Pegando a máxima e a mínima do período futuro
df_model = df_feat.withColumn("max_high_fwd", f.max("high").over(w_forward)) \
                  .withColumn("min_low_fwd", f.min("low").over(w_forward))

# Lógica das 3 Barreiras
df_model = df_model.withColumn(
    "target",
    f.when(
        # Condição de Sucesso: Bateu no Take Profit e NÃO bateu no Stop Loss no mesmo horizonte
        (f.col("max_high_fwd") >= f.col("close") * (1 + threshold)) & 
        (f.col("min_low_fwd") > f.col("close") * (1 - stop_loss)),
        1
    ).otherwise(0) # Falha: Bateu no stop loss ou o tempo expirou sem bater no alvo
)

# Remove as colunas auxiliares e os nulos gerados pela janela
df_model = df_model.drop("max_high_fwd", "min_low_fwd").dropna()

In [0]:
# remove nulos gerados pelas janelas e pelo lead do target
df_model = df_model.dropna()

# converte o sinal categórico para numérico
df_model = df_model.withColumn(
    "vol_signal_idx", 
    f.when(f.col("vol_signal") == "Normal", 0)
    .when(f.col("vol_signal") == "Forte compra (baleia)", 1)
    .otherwise(2) # Forte venda (dump)
)
df_model.limit(5).display()

In [0]:
df_model.groupBy("target").count().show()

| target | count |
|--------|-------|
|   0    |  902  |
|   1    |   97  |

as classes estão desbalanceadas
- threshhold está muito alto?

In [0]:
df_model = df_model.toPandas()

In [0]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

# Momentum de 3 períodos (Variação da variação)
df_model['momentum'] = (df_model['close'] - df_model['close'].shift(3)) / df_model['close'].shift(3)
# Hora do dia (Feature Categórica)
df_model['hour'] = pd.to_datetime(df_model['timestamp']).dt.hour

features_cols = [
    "log_return"
    # ,"rsi"
    ,"vol_change"
    # ,"z_score_vol" 
    ,"z_score_close"
    # ,"assimetria"
    # ,"curtosi"
    ,"dist_VWAP"
    # , "vol_signal_idx"
    ,"momentum"
    ,"hour"
    ,"btc_log_return"
    ,"forca_relativa_btc"
]

df_final = df_model[features_cols + ["target", "timestamp"]].dropna().sort_values("timestamp")

# assembler (Transforma colunas em um único vetor 'features')
X = df_final[features_cols]
y = df_final["target"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [0]:
# ordenar por timestamp (garantia)
df_final = df_final.sort_values("timestamp")

# Divisão simples de Treino e Teste (SEM Undersampling)
split_idx = int(len(df_final) * 0.8)
train_df = df_final.iloc[:split_idx]
test_df = df_final.iloc[split_idx:]

X_train = train_df[features_cols]
y_train = train_df["target"]
X_test = test_df[features_cols]
y_test = test_df["target"]

print("Distribuição real no treino:\n", train_df.value_counts("target"))

In [0]:
!pip install xgboost

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Parâmetros de risco/retorno
REWARD = 2.0  # Unidades ganhas no Take Profit
RISK = 1.0    # Unidades perdidas no Stop Loss
FEE = 0.15    # Taxa da corretora + Slippage estimado por trade

# arrays para armazenar os resultados
thresholds = np.linspace(0.30, 0.70, 81) # Testando de 0.30 a 0.70 com passos de 0.005
ev_per_trade_list = []
total_ev_list = []
win_rates = []
total_trades = []

for thresh in thresholds:
    # Simula as previsões com o threshold atual
    preds = (probabilities >= thresh).astype(int)
    
    # Se não houver nenhum sinal de compra, zera os stats para evitar divisão por zero
    if np.sum(preds) == 0:
        ev_per_trade_list.append(0)
        total_ev_list.append(0)
        win_rates.append(0)
        total_trades.append(0)
        continue

    # matriz de confusão
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    
    trades_realizados = tp + fp
    
    # se gerou menos de 10 trades na base de teste inteira, consideramos irrelevante (overfitting/sorte)
    if trades_realizados < 10:
        ev_per_trade_list.append(0)
        total_ev_list.append(0)
        win_rates.append(0)
        total_trades.append(0)
        continue

    # Win Rate (Precision)
    win_rate = tp / trades_realizados
    loss_rate = fp / trades_realizados
    
    # Cálculo do Expected Value (EV) por trade, descontando as taxas em todas as operações
    ev_trade = (win_rate * (REWARD - FEE)) - (loss_rate * (RISK + FEE))
    
    # Cálculo do EV Total (Lucro bruto projetado no período de teste)
    ev_total = ev_trade * trades_realizados
    
    ev_per_trade_list.append(ev_trade)
    total_ev_list.append(ev_total)
    win_rates.append(win_rate)
    total_trades.append(trades_realizados)

# Identificando o melhor threshold baseado no EV Total
best_idx = np.argmax(total_ev_list)
best_thresh = thresholds[best_idx]
best_ev_total = total_ev_list[best_idx]
best_ev_trade = ev_per_trade_list[best_idx]

print(f"--- Calibração Concluída ---")
print(f"Melhor Threshold Matemático: {best_thresh:.3f}")
print(f"Win Rate neste ponto: {win_rates[best_idx]*100:.2f}%")
print(f"Total de Trades disparados: {total_trades[best_idx]}")
print(f"EV por Trade (Lucro médio esperado): +{best_ev_trade:.3f} R")
print(f"EV Total no período de teste: +{best_ev_total:.2f} R")

# Plotando a Curva de Lucratividade
plt.figure(figsize=(12, 6))

# Plot EV Total
ax1 = plt.gca()
line1, = ax1.plot(thresholds, total_ev_list, color='blue', linewidth=2, label='Lucro Total Projetado (EV)')
ax1.set_xlabel('threshold proba', fontsize=12)
ax1.set_ylabel('Lucro total (Unidades de Risco)', color='blue', fontsize=12)
ax1.axvline(x=best_thresh, color='red', linestyle='--', label=f'Threshold Ótimo ({best_thresh:.2f})')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=1)

# Plot Qtde de Trades no eixo secundário (para entender a liquidez do sinal)
ax2 = ax1.twinx()
line2, = ax2.plot(thresholds, total_trades, color='grey', linestyle=':', alpha=0.7, label='Qtd de Trades')
ax2.set_ylabel('Total de Trades', color='grey', fontsize=12)

# Juntando as legendas
lines = [line1, line2, ax1.lines[0]] # line1, line2 e axvline
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper right')

plt.title('Otimização de Threshold por Expectativa Matemática (Descontando Taxas)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

abaixo de 0.50, as taxas da corretora e os falsos positivos (overtrading) destroem a conta. A partir de 0.51, o modelo cruza a linha do zero e passa a ter vantagem estatística, atingindo o pico de eficiência em **0.595**

In [0]:
import xgboost as xgb

# Calcula o peso dinâmico para lidar com o desbalanceamento naturalmente
contagem_classes = y_train.value_counts()
ratio = float(contagem_classes[0] / contagem_classes[1])

# Configuração e Treinamento do XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=4,           # Evita overfitting em ruído
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio, # substitui o undersampling
    random_state=42,
    eval_metric='logloss'
)

model = xgb_model.fit(X_train, y_train)

# Fazendo as previsões com o melhor threshold
probabilities = model.predict_proba(X_test)[:, 1]
predictions = (probabilities >= best_thresh).astype(int)


In [0]:
import sklearn.metrics as m

# Avaliação das métricas
auc         = m.roc_auc_score(y_test, probabilities)
accuracy    = m.accuracy_score(y_test, predictions)
f1          = m.f1_score(y_test, predictions)
precision   = m.precision_score(y_test, predictions)
recall      = m.recall_score(y_test, predictions)
cm          = m.confusion_matrix(y_test, predictions)

print(f"Área sob a Curva ROC (AUC): {auc:.4f}")
print(f"Acurácia Geral: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("Confusion Matrix:")
print(cm)

# previsões
pred_df = test_df.copy()
pred_df["prediction"] = predictions
pred_df["probability"] = probabilities
display(pred_df[["timestamp", "target", "prediction", "probability"]].head(10))

- att0: Apesar da acurácia estar aparentemente boa (81%), a curva ROC está dizendo que o modelo não está conseguindo distinguir entre as classes, com um AUC de 50% — equivalente a uma escolha aleatória (como jogar uma moeda para decidir se o mercado vai subir ou descer)

- att1: acuracia abaixou para 49% após undersampling, porém, roc abaixou tambem! (44%)

- att2: acurácia continua 49%, porém curva roc foi para 55%. isso é bacana. ocorreu após utilizar o threshhold dinâmico. irei focar em achar o threshold ideal

In [0]:
import joblib
# Exporta o modelo para .pkl
joblib.dump(model, "model/xgb_model.pkl")

In [0]:
# probabilidade da classe 1 (subida)
probabilities = model.predict_proba(X_test)[:, 1]

# em vez de usar o padrão 0.5, vou baixar o threshold
# se o modelo tiver x% de certeza que vai subir, nós consideramos como sinal.
novo_threshold = 0.595 
custom_predictions = (probabilities >= novo_threshold).astype(int)

# recalcular métricas com o novo threshold
from sklearn.metrics import classification_report, confusion_matrix

print(f"--- Métricas com Threshold de {novo_threshold} ---")
print(confusion_matrix(y_test, custom_predictions))
print(classification_report(y_test, custom_predictions))

# AUC deve permanecer a mesma (pois ela avalia a probabilidade, não a classe fixa)
auc = m.roc_auc_score(y_test, probabilities)
print(f"AUC Inalterada: {auc:.4f}")

fuck!

In [0]:
len(predictions)

In [0]:
import pandas as pd
# extraindo a importância
importances = model.feature_importances_
feat_imp = pd.DataFrame(list(zip(features_cols, importances)), 
                        columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)
feat_imp

tem que desconsiderar algumas variáveis inúteis